# 00 Finetune v1 Colab - CLRKDNet Field1+Field2

이 노트북은 **한 번 Run all로 끝내는 노트북이 아님.**

목표는 공식 `ResNet18_CULane.py` 학습 로직을 최대한 유지한 상태에서, `ResNet18_CULane.pth`를 field1+field2 pseudo CULane dataset으로 fine-tuning하는 **v1 baseline**을 만드는 것임.

v1 실험 조건:

```text
base checkpoint: ResNet18_CULane.pth
train dataset: map_culane_field1_field2_train_component_poly
holdout dataset: map_culane_holdout_eval_component_poly
config base: official configs/ResNet18_CULane.py
lr: 1e-4
epochs: 12
augmentation: official ResNet18_CULane train_process 유지
```

실행 방식:

```text
1. 준비물/데이터 확인
2. 공식 repo/env 세팅
3. config patch 결과 확인
4. smoke 1 epoch 실행
5. smoke 결과를 보고 진행 여부 판단
6. full fine-tuning 실행
7. 결과 Drive 저장
```

중간 결과가 이상하면 그 지점에서 멈추고 수정해야 함.


## 공식 repo 기준으로 무엇을 그대로 쓰고, 무엇을 바꾸는가

이 노트북은 CLRKDNet 논문의 teacher-student distillation 전체를 처음부터 재현하는 노트북이 아님. 목표는 이미 공개된 **CLRKDNet ResNet18 CULane pretrained checkpoint**를 우리 맵 도메인에 맞게 **supervised fine-tuning**하는 것임.

공식 repo/논문 기준에서 유지하는 것:

- `Detector` / `ResNetWrapper(resnet18)` / `Aggregator` / `CLRHead` 구조
- `num_priors=192`, `refine_layers=1`, `sample_points=36`
- CULane dataset loader가 기대하는 `list/*`, `*.lines.txt`, `laneseg_label_w16` 구조
- `GenerateLaneLine -> ToTensor` 학습 pipeline
- `cls_loss`, `reg_xytl_loss`, `iou_loss`, `seg_loss` 조합
- `main.py ... --finetune_from ResNet18_CULane.pth` 로 checkpoint를 이어받는 방식

우리 맵 때문에 바꾸는 것:

- 원본 이미지 geometry: `1640x590` -> `1296x972`
- crop: `cut_height=270` -> `cut_height=445`
- `sample_y`: CULane y 범위 -> 우리 카메라 crop 이후 하단 도로 영역
- dataset path: `./data/CULane` -> `./data/map_culane_field1_field2_train_component_poly`
- `diff_path`: CULane similarity-removal npz 없음 -> `None`
- batch/epoch/lr: Colab T4와 우리 데이터 크기에 맞게 조정

중요한 판단:

- `--distillation`은 사용하지 않음. teacher model까지 새로 학습하는 것이 아니라, 이미 distillation으로 만들어진 student checkpoint를 우리 맵에 fine-tuning하는 단계이기 때문임.
- NMS CUDA extension은 validation/postprocess 쪽 이슈라서, Colab 호환성 문제를 피하기 위해 top-k fallback을 둠. 학습 loss 계산 자체에는 custom NMS가 직접 들어가지 않음.


## 0. Colab 실행 전 준비물

Google Drive에 아래 폴더를 만들고, 파일 3개를 올려둠.

```text
MyDrive/CLRKDNet_Field1Field2_Finetune/
├─ ResNet18_CULane.pth
├─ map_culane_field1_field2_train_component_poly.tar.gz
└─ map_culane_holdout_eval_component_poly.tar.gz
```

로컬 기준 원본 위치는 다음과 같았음.

```text
20_shared_assets/models/model/ResNet18_CULane.pth
30_pipelines/culane_pseudo_dataset_builder/archives/map_culane_field1_field2_train_component_poly.tar.gz
30_pipelines/culane_pseudo_dataset_builder/archives/map_culane_holdout_eval_component_poly.tar.gz
```

Colab 런타임은 가능하면 **T4 GPU**로 선택함.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, tarfile, textwrap, subprocess, sys

DRIVE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/26-1학기_임베디드인공지능시스템최적화/03_CLRKDNet_Fine-Tuning_v2')
REPO_DIR = Path('/content/CLRKDNet')
DATA_ROOT = REPO_DIR / 'data'
TRAIN_DATASET_NAME = 'map_culane_field1_field2_train_component_poly'
HOLDOUT_DATASET_NAME = 'map_culane_holdout_eval_component_poly'
TRAIN_TAR = DRIVE_DIR / f'{TRAIN_DATASET_NAME}.tar.gz'
HOLDOUT_TAR = DRIVE_DIR / f'{HOLDOUT_DATASET_NAME}.tar.gz'
CKPT_SRC = DRIVE_DIR / 'ResNet18_CULane.pth'
OUT_DRIVE = DRIVE_DIR / 'outputs'

print('DRIVE_DIR:', DRIVE_DIR)
print('DRIVE_DIR exists:', DRIVE_DIR.exists())
for p in [CKPT_SRC, TRAIN_TAR, HOLDOUT_TAR]:
    print(p.name, 'exists=', p.exists(), 'size_MB=', round(p.stat().st_size / 1024 / 1024, 2) if p.exists() else None)

if not DRIVE_DIR.exists():
    raise FileNotFoundError(f'Drive folder not found: {DRIVE_DIR}')
missing = [p.name for p in [CKPT_SRC, TRAIN_TAR, HOLDOUT_TAR] if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required files in DRIVE_DIR: ' + ', '.join(missing))


## 1. CLRKDNet repo 준비

공식 repo를 `/content/CLRKDNet`에 clone함. Colab 런타임을 재실행했을 때 이미 clone되어 있으면 재사용함.


In [ ]:
if not REPO_DIR.exists():
    !git clone https://github.com/weiqingq/CLRKDNet.git /content/CLRKDNet
else:
    print('repo already exists:', REPO_DIR)


## 2. Dependency 설치

공식 repo는 Ubuntu/Python 3.8/PyTorch 1.x/CUDA 10.x 환경을 기준으로 작성되어 있음. 현재 Colab 런타임은 이보다 최신이라 `mmcv`, custom CUDA NMS, `imgaug` 쪽에서 호환성 문제가 생길 수 있음.

여기서는 학습에 필요한 최소 호환성 patch만 적용함.

- `mmcv` shim: repo 내부에서 사용하는 `ConvModule`, `MMDataParallel`, `DataContainer`, `collate` 최소 구현
- NMS fallback: validation/get_lanes용 top-k fallback
- NumPy/imgaug compatibility patch

이 patch들은 모델 구조나 loss 수식을 바꾸는 것이 아니라, 현재 Colab 환경에서 공식 repo의 학습 루프를 실행하기 위한 호환성 처리임.


In [ ]:
!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q "numpy==1.26.4" "opencv-python-headless==4.10.0.84"
!python -m pip install -q addict yapf pathspec tqdm pandas scikit-image pillow
!python -m pip install -q timm pytorch_warmup ptflops
!python -m pip install -q "imgaug==0.4.0"
!python -m pip install -q "shapely>=2.0.0" p_tqdm


In [ ]:
import importlib, sys
mods = ['torch', 'cv2', 'numpy', 'pandas', 'imgaug', 'shapely', 'timm', 'pytorch_warmup']
for name in mods:
    try:
        mod = importlib.import_module(name)
        print(name, 'OK', getattr(mod, '__version__', ''))
    except Exception as e:
        print(name, 'FAIL', type(e).__name__, str(e)[:200])
print('python:', sys.version)


In [ ]:
# Compatibility patches for current Colab.
# 1) Provide a tiny local mmcv shim: ConvModule, jit, MMDataParallel, DataContainer, collate.
# 2) Replace custom CUDA NMS import with a safe top-k fallback for smoke/validation.
# 3) Patch collections.Iterable and imgaug/NumPy compatibility.
from pathlib import Path

mmcv_root = REPO_DIR / 'mmcv'
(mmcv_root / 'cnn').mkdir(parents=True, exist_ok=True)
(mmcv_root / 'parallel').mkdir(parents=True, exist_ok=True)

mmcv_init_code = '''
__version__ = 'local-shim'

def jit(*jit_args, **jit_kwargs):
    def decorator(func):
        return func
    if len(jit_args) == 1 and callable(jit_args[0]) and not jit_kwargs:
        return jit_args[0]
    return decorator

def load(filename, *args, **kwargs):
    import json
    with open(filename, 'r') as f:
        return json.load(f)

def dump(obj, file=None, file_format=None, *args, **kwargs):
    import json
    text = json.dumps(obj, indent=2)
    if file is None:
        return text
    with open(file, 'w') as f:
        f.write(text)
'''
(mmcv_root / '__init__.py').write_text(mmcv_init_code, encoding='utf-8')

conv_module_code = '''
import torch.nn as nn

class ConvModule(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0,
                 dilation=1, groups=1, bias='auto', conv_cfg=None, norm_cfg=None,
                 act_cfg=dict(type='ReLU'), inplace=True, **kwargs):
        super().__init__()
        if bias == 'auto':
            bias = norm_cfg is None
        self.with_norm = norm_cfg is not None
        self.with_activation = act_cfg is not None
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride,
                              padding=padding, dilation=dilation, groups=groups, bias=bias)
        if self.with_norm:
            self.bn = nn.BatchNorm2d(out_channels)
        if self.with_activation:
            act_type = act_cfg.get('type', 'ReLU') if isinstance(act_cfg, dict) else 'ReLU'
            if act_type == 'ReLU':
                self.activate = nn.ReLU(inplace=inplace)
            elif act_type == 'LeakyReLU':
                self.activate = nn.LeakyReLU(inplace=inplace)
            else:
                self.activate = nn.ReLU(inplace=inplace)

    def forward(self, x):
        x = self.conv(x)
        if self.with_norm:
            x = self.bn(x)
        if self.with_activation:
            x = self.activate(x)
        return x
'''
(mmcv_root / 'cnn' / '__init__.py').write_text(conv_module_code, encoding='utf-8')

parallel_code = '''
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data._utils.collate import default_collate

class DataContainer:
    def __init__(self, data, cpu_only=False):
        self.data = data
        self.cpu_only = cpu_only

class MMDataParallel(nn.Module):
    def __init__(self, module, device_ids=None):
        super().__init__()
        self.module = module
        self.device_ids = device_ids

    def cuda(self, device=None):
        self.module.cuda(device)
        return self

    def forward(self, *args, **kwargs):
        return self.module(*args, **kwargs)

def collate(batch, samples_per_gpu=1):
    first = batch[0]
    if isinstance(first, dict):
        out = {}
        for key in first.keys():
            vals = [sample[key] for sample in batch]
            v0 = vals[0]
            if isinstance(v0, DataContainer):
                out[key] = DataContainer([v.data for v in vals], cpu_only=v0.cpu_only)
            elif torch.is_tensor(v0):
                out[key] = torch.stack(vals, dim=0)
            elif isinstance(v0, np.ndarray):
                out[key] = torch.from_numpy(np.stack(vals, axis=0))
            elif isinstance(v0, (int, float, np.integer, np.floating)):
                out[key] = torch.tensor(vals)
            else:
                out[key] = vals
        return out
    return default_collate(batch)
'''
(mmcv_root / 'parallel' / '__init__.py').write_text(parallel_code, encoding='utf-8')

# NMS fallback. For training, NMS is not part of loss. It is only needed by validation/get_lanes.
nms_py = REPO_DIR / 'clrkd' / 'ops' / 'nms.py'
nms_py.write_text('''
try:
    from . import nms_impl
except Exception:
    nms_impl = None

import torch

def nms(boxes, scores, overlap, top_k):
    if nms_impl is not None:
        return nms_impl.nms_forward(boxes, scores, overlap, top_k)
    order = torch.argsort(scores, descending=True)
    keep = order[:top_k].contiguous().long()
    return keep, keep.numel(), None
''', encoding='utf-8')

# Python 3.10+ compatibility.
transforms_py = REPO_DIR / 'clrkd' / 'datasets' / 'process' / 'transforms.py'
text = transforms_py.read_text()
text = text.replace('collections.Iterable', 'collections.abc.Iterable')
transforms_py.write_text(text)

# imgaug 0.4.0 expects np.sctypes in some environments.
gll_py = REPO_DIR / 'clrkd' / 'datasets' / 'process' / 'generate_lane_line.py'
text = gll_py.read_text()
if 'np.sctypes = {' not in text:
    insert = (
        'import numpy as np\n\n'
        'if not hasattr(np, "sctypes"):\n'
        '    np.sctypes = {\n'
        '        "int": [np.int8, np.int16, np.int32, np.int64],\n'
        '        "uint": [np.uint8, np.uint16, np.uint32, np.uint64],\n'
        '        "float": [np.float16, np.float32, np.float64],\n'
        '        "complex": [np.complex64, np.complex128],\n'
        '        "others": [np.bool_, np.bytes_, np.str_],\n'
        '    }\n'
    )
    text = text.replace('import numpy as np\n', insert, 1)
gll_py.write_text(text)

print('compatibility patches written')
print('mmcv shim:', mmcv_root)
print('nms fallback:', nms_py)
print('imgaug numpy patch:', gll_py)


## 3. 최종 train/holdout dataset 배치

학습에는 `map_culane_field1_field2_train_component_poly`를 사용함. Holdout archive는 이 노트북에서 학습에 쓰지는 않지만, 같은 Drive output 폴더에 보관해두고 다음 노트북에서 pretrained/fine-tuned 비교에 사용함.

압축 파일에는 review overlay까지 포함되어 있을 수 있음. 학습에는 `list/`, image, `*.lines.txt`, `laneseg_label_w16/`만 필요하므로, Colab 디스크가 부족하면 `_review_overlays`를 삭제함.


In [ ]:
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def extract_tar_if_needed(tar_path: Path, expected_dir: Path):
    if expected_dir.exists():
        print('already extracted:', expected_dir)
        return
    print('extracting:', tar_path.name)
    with tarfile.open(tar_path, 'r:gz') as tf:
        tf.extractall(DATA_ROOT)
    print('done:', expected_dir, 'exists=', expected_dir.exists())

TRAIN_DATA_DIR = DATA_ROOT / TRAIN_DATASET_NAME
HOLDOUT_DATA_DIR = DATA_ROOT / HOLDOUT_DATASET_NAME
extract_tar_if_needed(TRAIN_TAR, TRAIN_DATA_DIR)
extract_tar_if_needed(HOLDOUT_TAR, HOLDOUT_DATA_DIR)

# Save disk space: overlays are for local review, not for CLRKDNet training.
for d in [TRAIN_DATA_DIR / '_review_overlays', HOLDOUT_DATA_DIR / '_review_overlays']:
    if d.exists():
        shutil.rmtree(d)
        print('removed review overlays:', d)

shutil.copy(CKPT_SRC, REPO_DIR / 'ResNet18_CULane.pth')
shutil.rmtree(REPO_DIR / 'cache', ignore_errors=True)

for name, d in [('train', TRAIN_DATA_DIR), ('holdout', HOLDOUT_DATA_DIR)]:
    print('\n', name, d)
    print('exists:', d.exists())
    print('files:', sum(1 for _ in d.rglob('*') if _.is_file()))
    for summary_name in ['build_summary.json', 'validation_summary.json']:
        p = d / summary_name
        if p.exists():
            print(summary_name, json.loads(p.read_text()) if p.stat().st_size < 20000 else 'exists')


In [ ]:
# Check list sizes. These numbers should roughly match the final builder result.
def count_lines(path: Path):
    return sum(1 for _ in path.open('r', encoding='utf-8')) if path.exists() else None

TRAIN_LIST = TRAIN_DATA_DIR / 'list' / 'train_gt.txt'
VAL_LIST = TRAIN_DATA_DIR / 'list' / 'val.txt'
TEST_LIST = TRAIN_DATA_DIR / 'list' / 'test.txt'
HOLDOUT_TEST_LIST = HOLDOUT_DATA_DIR / 'list' / 'test.txt'

for p in [TRAIN_LIST, VAL_LIST, TEST_LIST, HOLDOUT_TEST_LIST]:
    print(p, 'count=', count_lines(p))


## STOP-CHECK A - Dataset 구조 확인 후 진행

여기서 잠깐 멈춤.

위 셀에서 아래가 정상인지 확인함.

- `train.txt`, `val.txt`, `test.txt` count가 출력됨.
- train dataset 압축이 제대로 풀림.
- `build_summary.json`, `validation_summary.json`가 읽힘.
- Drive 경로가 틀리지 않음.

이 단계에서 파일 없음, 압축 실패, count가 `None`이면 full 학습으로 넘어가지 않음.


## 4. Map fine-tuning config 생성

여기서는 config를 손으로 새로 쓰지 않고, 공식 repo의 `configs/ResNet18_CULane.py`를 읽은 뒤 **map adaptation에 필요한 항목만 patch**함.

이렇게 하는 이유는 다음과 같음.

- 공식 모델 구조와 학습 pipeline을 최대한 그대로 유지함.
- 우리 맵 때문에 바꾼 값이 무엇인지 명확히 남김.
- 이후 문제가 생겼을 때 “공식 구현 문제인지, map adaptation 설정 문제인지” 분리해서 볼 수 있음.

학습률은 우선 `1e-4`로 둠. 공식 config에는 batch size 8일 때 `3e-4`가 주석으로 언급되어 있지만, 우리는 pseudo-label 기반 도메인 fine-tuning이므로 첫 본 학습은 보수적으로 시작함. 필요하면 다음 실험에서 `3e-4` variant를 따로 비교함.


In [ ]:
import re

CONFIG_DIR = REPO_DIR / 'configs'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
BASE_CONFIG = CONFIG_DIR / 'ResNet18_CULane.py'
assert BASE_CONFIG.exists(), BASE_CONFIG

TRAIN_ROWS = count_lines(TRAIN_LIST)
BATCH_SIZE = 8
FULL_EPOCHS = 12
SMOKE_EPOCHS = 1
LR = 1.0e-4
TOTAL_ITER_FULL = max(1, (TRAIN_ROWS // BATCH_SIZE) * FULL_EPOCHS)
TOTAL_ITER_SMOKE = max(1, (TRAIN_ROWS // BATCH_SIZE) * SMOKE_EPOCHS)

base_text = BASE_CONFIG.read_text(encoding='utf-8')

def replace_assignment(text: str, name: str, value: str) -> str:
    pattern = rf'^{name}\s*=.*$'
    new_text, n = re.subn(pattern, f'{name} = {value}', text, flags=re.MULTILINE)
    if n != 1:
        raise RuntimeError(f'expected one assignment for {name}, replaced {n}')
    return new_text

def replace_dataset_split(text: str) -> str:
    # Official config uses val split='test' for CULane validation. Our generated dataset has explicit val/test lists.
    old = "val=dict(\n    type=dataset_type,\n    data_root=dataset_path,\n    split='test',"
    new = "val=dict(\n    type=dataset_type,\n    data_root=dataset_path,\n    split='val',"
    if old in text:
        text = text.replace(old, new, 1)
    return text

def build_map_config(*, epochs, total_iter, eval_ep, save_ep, work_dirs):
    text = base_text
    text = replace_assignment(text, 'sample_y', 'range(971, 444, -20)')
    text = replace_assignment(text, 'epochs', str(epochs))
    text = replace_assignment(text, 'batch_size', str(BATCH_SIZE))
    text = replace_assignment(text, 'eval_ep', str(eval_ep))
    text = replace_assignment(text, 'save_ep', str(save_ep))
    text = replace_assignment(text, 'optimizer', f"dict(type='AdamW', lr={LR})")
    text = replace_assignment(text, 'total_iter', str(total_iter))
    text = replace_assignment(text, 'test_parameters', "dict(conf_threshold=0.35, nms_thres=50, nms_topk=max_lanes)")
    text = replace_assignment(text, 'work_dirs', repr(work_dirs))
    text = replace_assignment(text, 'ori_img_w', '1296')
    text = replace_assignment(text, 'ori_img_h', '972')
    text = replace_assignment(text, 'img_w', '800')
    text = replace_assignment(text, 'img_h', '320')
    text = replace_assignment(text, 'cut_height', '445')
    text = replace_assignment(text, 'dataset_path', repr(f'./data/{TRAIN_DATASET_NAME}'))
    text = replace_assignment(text, 'diff_path', 'None')
    text = replace_assignment(text, 'workers', '2')
    text = replace_assignment(text, 'log_interval', '20')
    text = replace_assignment(text, 'num_classes', 'max_lanes + 1')
    text = replace_dataset_split(text)

    # We load the whole detector from ResNet18_CULane.pth. Avoid accidental ImageNet weight download/init noise.
    text = text.replace(
        "backbone = dict(\n    type='ResNetWrapper',\n    resnet='resnet18',\n    pretrained=True)",
        "backbone = dict(\n    type='ResNetWrapper',\n    resnet='resnet18',\n    pretrained=False)",
    )

    if 'seed = 0' not in text:
        text = text.replace('# seed = 0', 'seed = 0')

    header = (
        '# Auto-generated from official configs/ResNet18_CULane.py\n'
        '# Only map geometry, dataset path, runtime schedule, and finetune-safe backbone init are patched.\n'
        f'# TRAIN_ROWS={TRAIN_ROWS}, BATCH_SIZE={BATCH_SIZE}, LR={LR}\n\n'
    )
    return header + text

CONFIG_SMOKE = CONFIG_DIR / 'MapLane_Field1Field2_ResNet18_smoke.py'
CONFIG_FULL = CONFIG_DIR / 'MapLane_Field1Field2_ResNet18_full.py'
CONFIG_SMOKE.write_text(build_map_config(
    epochs=SMOKE_EPOCHS,
    total_iter=TOTAL_ITER_SMOKE,
    eval_ep=1,
    save_ep=1,
    work_dirs='work_dirs/MapLane_Field1Field2_smoke',
), encoding='utf-8')
CONFIG_FULL.write_text(build_map_config(
    epochs=FULL_EPOCHS,
    total_iter=TOTAL_ITER_FULL,
    eval_ep=999,
    save_ep=2,
    work_dirs='work_dirs/MapLane_Field1Field2_full',
), encoding='utf-8')

print('BASE_CONFIG:', BASE_CONFIG)
print('CONFIG_SMOKE:', CONFIG_SMOKE)
print('CONFIG_FULL:', CONFIG_FULL)
print('TRAIN_ROWS:', TRAIN_ROWS)
print('TOTAL_ITER_SMOKE:', TOTAL_ITER_SMOKE)
print('TOTAL_ITER_FULL:', TOTAL_ITER_FULL)
print('\nPatched config preview:')
for line in CONFIG_FULL.read_text(encoding='utf-8').splitlines()[:55]:
    print(line)


## STOP-CHECK B - Config patch 확인 후 진행

여기서 잠깐 멈춤.

`Patched config preview`에서 최소한 아래 값들이 맞는지 확인함.

```text
sample_y = range(971, 444, -20)
ori_img_w = 1296
ori_img_h = 972
img_w = 800
img_h = 320
cut_height = 445
dataset_path = './data/map_culane_field1_field2_train_component_poly'
diff_path = None
optimizer lr = 1e-4
```

이 값이 다르면 공식 config patch가 잘못된 것이므로 smoke 학습으로 넘어가지 않음.


## 5. Validation geometry patch

공식 CULane 코드는 평가 시 `590 x 1640` 기준이 하드코딩된 부분이 있음. 우리는 원본 이미지가 `1296 x 972`이고 `cut_height=445`이므로, validation 시각화/metric 계산에서 map geometry를 쓰도록 patch함.


In [ ]:
culane_py = REPO_DIR / 'clrkd' / 'datasets' / 'culane.py'
text = culane_py.read_text()
text = text.replace(
    "ys = np.arange(270, 590, 8) / self.cfg.ori_img_h",
    "ys = np.arange(self.cfg.cut_height, self.cfg.ori_img_h, 8) / self.cfg.ori_img_h",
)
text = text.replace(
    "culane_metric.eval_predictions(output_basedir,\n                                                    self.data_root,\n                                                    os.path.join(self.data_root, cate_file),\n                                                    iou_thresholds=[0.5],\n                                                    official=True)",
    "culane_metric.eval_predictions(output_basedir,\n                                                    self.data_root,\n                                                    os.path.join(self.data_root, cate_file),\n                                                    iou_thresholds=[0.5],\n                                                    official=True,\n                                                    img_shape=(self.cfg.ori_img_h, self.cfg.ori_img_w, 3))",
)
text = text.replace(
    "culane_metric.eval_predictions(output_basedir,\n                                                self.data_root,\n                                                self.list_path,\n                                                iou_thresholds=np.linspace(0.5, 0.95, 10),\n                                                official=True)",
    "culane_metric.eval_predictions(output_basedir,\n                                                self.data_root,\n                                                self.list_path,\n                                                iou_thresholds=np.linspace(0.5, 0.95, 10),\n                                                official=True,\n                                                img_shape=(self.cfg.ori_img_h, self.cfg.ori_img_w, 3))",
)
culane_py.write_text(text)

metric_py = REPO_DIR / 'clrkd' / 'utils' / 'culane_metric.py'
text = metric_py.read_text()
text = text.replace(
    "def eval_predictions(pred_dir,\n                     anno_dir,\n                     list_path,\n                     iou_thresholds=[0.5],\n                     width=30,\n                     official=True,\n                     sequential=False):",
    "def eval_predictions(pred_dir,\n                     anno_dir,\n                     list_path,\n                     iou_thresholds=[0.5],\n                     width=30,\n                     official=True,\n                     sequential=False,\n                     img_shape=(590, 1640, 3)):",
)
text = text.replace("    img_shape = (590, 1640, 3)\n", "")
metric_py.write_text(text)
print('patched map validation geometry')


## 6. 1-epoch smoke fine-tuning

먼저 1 epoch만 돌림. 이 셀의 목적은 성능을 판단하는 것이 아니라 다음을 확인하는 것임.

- dataset 구조를 CLRKDNet이 읽을 수 있는지
- pretrained checkpoint가 정상 로드되는지
- loss가 계산되고 역전파가 되는지
- work_dir/checkpoint가 생성되는지


In [ ]:
%cd /content/CLRKDNet
!python main.py configs/MapLane_Field1Field2_ResNet18_smoke.py \
  --gpus 0 \
  --finetune_from /content/CLRKDNet/ResNet18_CULane.pth \
  --work_dirs work_dirs/MapLane_Field1Field2_smoke


In [ ]:
!find /content/CLRKDNet/work_dirs/MapLane_Field1Field2_smoke -maxdepth 4 -type f | sort | tail -80


## STOP-CHECK C - Smoke 학습 결과 확인 후 full 진행

여기서 반드시 멈춤.

Smoke 학습 결과에서 확인할 것:

- 에러 없이 종료됨.
- `loss`, `cls_loss`, `reg_xytl_loss`, `seg_loss`, `iou_loss`가 `nan`이 아님.
- `work_dirs/MapLane_Field1Field2_smoke` 아래 checkpoint/log가 생성됨.

이 중 하나라도 이상하면 full 학습을 실행하지 말고 결과를 확인해야 함.


## 7. Full fine-tuning

Smoke가 통과하면 full fine-tuning을 실행함. 이 노트북의 기본값은 `12 epochs`임. 너무 오래 걸리면 중간 checkpoint만 받아도 되고, 결과가 부족하면 이후 실험에서 epoch/lr를 조절함.

주의: 이 metric은 pseudo-label 기준이므로 절대적인 주행 성능이 아님. 최종 판단은 다음 노트북에서 holdout/background 이미지에 대한 시각 비교로 함.


In [ ]:
%cd /content/CLRKDNet
!python main.py configs/MapLane_Field1Field2_ResNet18_full.py \
  --gpus 0 \
  --finetune_from /content/CLRKDNet/ResNet18_CULane.pth \
  --work_dirs work_dirs/MapLane_Field1Field2_full


## 8. 결과를 Drive로 복사

다음 노트북에서 PC로 내려받아 분석할 수 있게 work_dirs, config, dataset summary를 Drive output 폴더로 복사함.


In [ ]:
OUT_DRIVE.mkdir(parents=True, exist_ok=True)

# Work dirs
for name in ['MapLane_Field1Field2_smoke', 'MapLane_Field1Field2_full']:
    src = REPO_DIR / 'work_dirs' / name
    dst = OUT_DRIVE / name
    if dst.exists():
        shutil.rmtree(dst)
    if src.exists():
        shutil.copytree(src, dst)
        print('copied:', src, '->', dst)
    else:
        print('missing work_dir:', src)

# Configs and summaries
meta_dir = OUT_DRIVE / 'run_inputs_and_summaries'
meta_dir.mkdir(parents=True, exist_ok=True)
for p in [CONFIG_SMOKE, CONFIG_FULL, TRAIN_DATA_DIR / 'build_summary.json', TRAIN_DATA_DIR / 'validation_summary.json', HOLDOUT_DATA_DIR / 'build_summary.json', HOLDOUT_DATA_DIR / 'validation_summary.json']:
    if p.exists():
        shutil.copy(p, meta_dir / p.name)
        print('copied meta:', p.name)

print('OUT_DRIVE:', OUT_DRIVE)


## 9. 이번 노트북에서 확인할 것

이 노트북은 다음 단계로 넘어가기 전, 아래 결과를 확인하기 위한 v1 baseline임.

확인할 것:

1. Drive 준비물 3개가 정확히 인식되는가.
2. train/val/test list count가 예상 범위로 나오는가.
3. 공식 config 기반 patch preview에서 map geometry가 정상인지 확인했는가.
4. smoke 1 epoch가 에러 없이 끝났는가.
5. smoke loss가 `nan`이 아니고 checkpoint가 생성되었는가.
6. full 학습 결과가 `outputs/MapLane_Field1Field2_full`로 복사되었는가.

이 노트북에서 아직 하지 않는 것:

- fine-tuned 모델이 실제로 더 좋은지 최종 판단
- pretrained vs fine-tuned holdout 시각 비교
- ONNX export
- PTQ 양자화
- Pi runtime 측정
- 주행 제어 후처리 튜닝

다음 노트북은 `01_training_result_review.ipynb` 또는 `01_holdout_visual_compare.ipynb`로 이어가면 됨.
